# Chisamba CDF Project EDA

This notebook explores the cleaned Chisamba Constituency Development Fund (CDF) project dataset by combining the approved project records for 2024 and 2025. The goal is to profile the project portfolio, identify the most common sectors and wards, and surface useful patterns for the group project deliverable.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')

repo_root = Path.cwd()
project_2024 = repo_root / 'db-unza26-csc4792-chisamba_cdf_projects.csv'
project_2025 = repo_root / 'db-unza26-csc4792-chisamba_2025_approved_projects (1).csv'

# Load the cleaned project datasets and combine them for a unified portfolio view.
df_2024 = pd.read_csv(project_2024, sep='|')
df_2025 = pd.read_csv(project_2025, sep='|')
df = pd.concat([df_2024, df_2025], ignore_index=True)
df['financial_year'] = df['financial_year'].astype(int)
print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]} | Columns: {df.shape[1]}')
display(df.head())


## 1. Data overview

The first step is to confirm the combined dataset structure, data types, and missingness before analyzing patterns. This helps detect issues such as inconsistent values, unparsed columns, or incomplete records.


In [ ]:
print('Column dtypes:
')
print(df.dtypes.to_string())
print('
Missing values by column:
')
print(df.isna().sum().to_string())
print(f'
Duplicate rows: {df.duplicated().sum()}')
print(f'Unique project IDs: {df['project_id'].nunique()}')


## 2. Missing values and data quality checks

The cleaned files are mostly complete, but a quick assessment is still necessary to validate data integrity and understand whether any fields were dropped or left blank.


In [ ]:
missing_summary = df.isna().sum().to_frame(name='missing_count')
missing_summary['missing_pct'] = (missing_summary['missing_count'] / len(df)) * 100
display(missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_count', ascending=False))

# Create a lightweight quality flag for project records with missing critical fields.
critical_cols = ['project_name', 'sector', 'ward', 'zone', 'financial_year']
df['has_missing_critical_fields'] = df[critical_cols].isna().any(axis=1)
print(f"Records with a missing critical field: {df['has_missing_critical_fields'].sum()}")


## 3. Univariate analysis

The first distribution checks focus on the project mix across sectors, wards, and years. This identifies where investment is concentrated and whether there are notable imbalances in planning.


In [ ]:
# Basic categorical breakdowns
sector_counts = df['sector'].value_counts().sort_values(ascending=False)
ward_counts = df['ward'].value_counts().head(10)
year_counts = df['financial_year'].value_counts().sort_index()

print('Projects by sector:
')
print(sector_counts.to_string())
print('
Top wards:
')
print(ward_counts.to_string())
print('
Projects by financial year:
')
print(year_counts.to_string())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sector_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Projects by Sector')
axes[0].set_xlabel('Sector')
axes[0].set_ylabel('Project Count')
axes[0].tick_params(axis='x', rotation=45)

ward_counts.plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Top Wards by Project Count')
axes[1].set_xlabel('Ward')
axes[1].set_ylabel('Project Count')
axes[1].tick_params(axis='x', rotation=45)

year_counts.plot(kind='bar', ax=axes[2], color='forestgreen')
axes[2].set_title('Projects by Financial Year')
axes[2].set_xlabel('Financial Year')
axes[2].set_ylabel('Project Count')

plt.tight_layout()
plt.show()

# Numeric-like text feature: project name length gives an indirect sense of project complexity.
df['project_name_length'] = df['project_name'].astype(str).str.len()
plt.figure(figsize=(8, 5))
sns.histplot(df['project_name_length'], bins=15, kde=True, color='mediumseagreen')
plt.title('Distribution of Project Name Length')
plt.xlabel('Project Name Length (characters)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


## 4. Outlier and spread checks

Because the dataset is mostly categorical, the most useful outlier check is based on counts per category rather than numeric fields. This highlights whether a few wards or sectors dominate the project portfolio.


In [ ]:
count_by_ward = df['ward'].value_counts()
count_by_sector = df['sector'].value_counts()

print('Ward count summary:')
print(count_by_ward.describe().to_string())
print('
Sector count summary:')
print(count_by_sector.describe().to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(y=count_by_ward.values, ax=axes[0], color='lightcoral')
axes[0].set_title('Distribution of Project Counts Across Wards')
axes[0].set_ylabel('Projects per Ward')

sns.boxplot(y=count_by_sector.values, ax=axes[1], color='cornflowerblue')
axes[1].set_title('Distribution of Project Counts Across Sectors')
axes[1].set_ylabel('Projects per Sector')

plt.tight_layout()
plt.show()


## 5. Bivariate and multivariate relationships

This section examines whether project allocation differs by year and sector, and whether certain wards or sectors dominate the portfolio. A heatmap is especially suitable for highlighting cross-tab patterns in this categorical dataset.


In [ ]:
# Cross-tabulation of sector by financial year.
year_sector_table = pd.crosstab(df['financial_year'], df['sector'])
display(year_sector_table)

plt.figure(figsize=(10, 6))
sns.heatmap(year_sector_table, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Project Sector Counts by Financial Year')
plt.xlabel('Sector')
plt.ylabel('Financial Year')
plt.tight_layout()
plt.show()

# Focus on the most active wards to compare distribution across sectors.
top_wards = df['ward'].value_counts().head(8).index
ward_sector_table = pd.crosstab(df[df['ward'].isin(top_wards)]['ward'], df['sector'])
plt.figure(figsize=(11, 7))
sns.heatmap(ward_sector_table, annot=True, fmt='d', cmap='Blues')
plt.title('Top Wards by Sector Mix')
plt.xlabel('Sector')
plt.ylabel('Ward')
plt.tight_layout()
plt.show()


## 6. Notable patterns and summary findings

The project portfolio is concentrated in a handful of sectors and wards, suggesting that investment is not evenly spread across the constituency. These patterns are important for the final report because they indicate where development effort is most visible and where planning may require broader geographic or sectoral diversification.

Potential takeaways:
- Sector mix indicates which development categories dominate the portfolio.
- Variations between 2024 and 2025 show whether project priorities shifted over time.
- Ward concentration highlights where activity is clustered, which matters for equity and resource planning.


In [ ]:
# Create a derived summary dataset for downstream reporting and dashboard use.
summary_df = (
    df.groupby(['financial_year', 'sector'], as_index=False)
      .size()
      .rename(columns={'size': 'project_count'})
      .sort_values(['financial_year', 'project_count'], ascending=[True, False])
)

summary_dir = repo_root / 'data' / 'processed'
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = summary_dir / 'eda_summary.csv'
summary_df.to_csv(summary_path, index=False)

print(summary_df.head(10).to_string(index=False))
print(f'\nSaved processed summary: {summary_path}')
